# ch02 文本表示（练习版）

> 补全 `____`，跑通所有 assert。真值全部实跑（sklearn 1.9.1）。
>
> `VOCAB` / `docs` / `vocab` / `idx` / 清洗与分词函数都已给好；
> 挖空块内保持**单行**，控制流都在块外。

In [ ]:
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

DATA = Path("data")
CSV = DATA / "synth_text.csv"

In [ ]:
df = pd.read_csv(CSV, keep_default_na=False)

# ---- 词表 / 停用词 / 正则（沿用 ch01）----
VOCAB = [
    "变压器", "断路器", "隔离开关", "避雷器", "电流互感器", "绝缘子", "套管", "母线",
    "渗油", "漏油", "油迹", "油污", "滴油", "油位下降",
    "锈蚀", "生锈", "铁锈", "漆面剥落", "镀锌层脱落",
    "破损", "裂纹", "碎裂", "缺口", "掉瓷", "箱体变形",
    "发热", "温度偏高", "过热", "烫手", "温升异常", "过热保护",
    "异物", "鸟巢", "塑料袋", "树枝", "风筝线搭挂",
    "发现", "巡检", "位于", "存在", "缺陷", "一般", "严重", "危急",
    "接报", "详见", "保护", "动作",
    "请安排抢修", "请尽快核实", "已通知运维班", "请列入本周计划",
    "需停电处理", "请核实后反馈",
    "研究", "研究生", "生命", "命", "起源",
]
VOCAB_SET = set(VOCAB)
MAX_WORD_LEN = max(len(w) for w in VOCAB)
SEG_RE = re.compile(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]+|[^\sA-Za-z0-9_\u4e00-\u9fff]")
HAN_RE = re.compile(r"[\u4e00-\u9fff]+")
STOPWORDS = {
    "的", "了", "在", "是", "和", "与", "及", "或", "也", "就", "都",
    "，", "。", "！", "？", "、", "（", "）", "【", "】", "：", "；", " ",
}
# 无效占位符：清洗救不了它们，只能做「有效性过滤」（ch01 §1.3 埋的伏笔）
BAD_TEXTS = {"", "N/A", "-", "无"}


def fmm(seg: str) -> list[str]:
    """正向最大匹配（ch01 手写过）。"""
    out, i = [], 0
    while i < len(seg):
        length = min(MAX_WORD_LEN, len(seg) - i)
        while length > 1 and seg[i:i + length] not in VOCAB_SET:
            length -= 1
        out.append(seg[i:i + length])
        i += length
    return out


def tokenize(text: str) -> list[str]:
    """两级切分：汉字段走最大匹配，其余整体入列。"""
    out = []
    for seg in SEG_RE.findall(text):
        out.extend(fmm(seg) if HAN_RE.fullmatch(seg) else [seg])
    return out


def clean(text: str) -> str:
    """五步清洗（ch01 手写过）。"""
    s = re.sub(r"<[^>]+>", "", text)
    s = re.sub(r"https?://\S+", "", s)
    s = s.replace(",", "，")
    s = re.sub(r"([，。！？])\1+", r"\1", s)
    s = re.sub(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])", "", s)
    return re.sub(r"\s+", " ", s).strip()


# ---- 有效性过滤：600 -> 570（这一步在 ch01 只是提了一句，本章必须做）----
texts = [clean(t) for t in df["缺陷描述"]]
keep = [i for i, t in enumerate(texts) if t.strip() not in BAD_TEXTS]
labels = df["缺陷类型"].to_numpy()[keep]
docs = [
    [w for w in tokenize(texts[i]) if w not in STOPWORDS and w.strip()]
    for i in keep
]

vocab = sorted({w for d in docs for w in d})
idx = {w: j for j, w in enumerate(vocab)}

print("原始工单", len(texts), "-> 有效", len(docs), "（剔掉无效占位符）")
print("词表大小", len(vocab), "| 平均每篇 token 数 %.4f" % (sum(len(d) for d in docs) / len(docs)))
print("文档数", len(docs), "| 标签分布", dict(Counter(map(str, labels))))

## 任务 1：手写词袋矩阵

坐标收集好了，你只需要把它累加进矩阵。

In [ ]:
pairs = [(i, idx[w]) for i, d in enumerate(docs) for w in d]
cbow = np.zeros((len(docs), len(vocab)), dtype=np.int32)

for r, c in pairs:
    # TODO(1)：把 (行, 列) 对应位置累加 1
    # 提示：cbow[r, c] += 1
    ____

print("词袋矩阵形状:", cbow.shape)
print("非零元素:", int((cbow > 0).sum()), "/", cbow.size)
print("第 0 篇的非零词:", [(vocab[j], int(cbow[0, j])) for j in np.nonzero(cbow[0])[0]])

# ---- 验收 ----
assert cbow.shape == (570, 70)
assert int(cbow.sum()) == 3431

## 任务 2：与 `CountVectorizer` 对账

注意 `token_pattern=None`。

In [ ]:
cv = CountVectorizer(tokenizer=lambda t: t, lowercase=False, token_pattern=None)
M_cv = cv.fit_transform(docs)

print("sklearn 形状:", M_cv.shape, "| 词表大小:", len(cv.vocabulary_))
print("与手写词表是否完全一致:", sorted(cv.vocabulary_) == vocab)

# ---- 验收 ----
assert M_cv.shape == cbow.shape
assert sorted(cv.vocabulary_) == vocab, "词表必须一字不差"
assert int(M_cv.sum()) == 3431

## 任务 3：稀疏度

零元素占比。

In [ ]:
n_cells = cbow.size
n_nonzero = int((cbow > 0).sum())

# TODO(2)：算稀疏度：零元素占比
# 提示：1 - 非零 / 总数
sparsity = ____

print("矩阵元素 %d 个，非零 %d 个" % (n_cells, n_nonzero))
print("稀疏度 %.6f" % sparsity)
print("如果词表涨到 5 万（真实中文词表量级），同样 570 篇文档的矩阵会到 %.1f 万个元素"
      % (570 * 50000 / 10000))

# ---- 验收 ----
assert abs(sparsity - 0.914010) < 1e-6

## 任务 4：教科书版 TF-IDF

`tf × ln(N/df)`，先不归一化。

In [ ]:
def tfidf_plain(docs, vocab):
    """教科书版 TF-IDF：tf * ln(N / df)，不做归一化。"""
    N = len(docs)
    doc_freq = Counter()
    for d in docs:
        for w in set(d):
            doc_freq[w] += 1

    M = np.zeros((len(docs), len(vocab)))
    for i, d in enumerate(docs):
        cnt = Counter(d)
        tf = {w: c / len(d) for w, c in cnt.items()}
        for w, t in tf.items():
            # TODO(3)：把 tf * ln(N / df) 写进第 i 行、第 idx[w] 列
            M[i, idx[w]] = ____
    return M


M_plain = tfidf_plain(docs, vocab)
print("教科书版形状:", M_plain.shape, "| 最大值 %.6f" % M_plain.max())

# ---- 验收 ----
assert M_plain.shape == (570, 70)

## 任务 5：对齐 sklearn 口径

两处：**平滑 idf** 的公式，以及**整行 L2 归一化**。
这一题做完，误差应该从 0.32 掉到 0。

In [ ]:
def tfidf_sklearn_like(docs, vocab):
    """对齐 sklearn 默认参数：smooth_idf=True + L2 归一化。"""
    N = len(docs)
    doc_freq = Counter()
    for d in docs:
        for w in set(d):
            doc_freq[w] += 1

    # TODO(4)：sklearn 的平滑 idf：ln((1 + N) / (1 + df)) + 1
    # 提示：注意分子分母都要 +1，最后再 +1
    idf = ____

    M = np.zeros((len(docs), len(vocab)))
    for i, d in enumerate(docs):
        cnt = Counter(d)
        cols = [idx[w] for w in cnt]
        # TODO(5)：先填 tf * idf，再对整行做 L2 归一化
        M[i, cols] = ____
        ____
    return M


M_sk = tfidf_sklearn_like(docs, vocab)
print("sklearn 口径版形状:", M_sk.shape, "| 行范数示例:",
      [round(float(np.linalg.norm(M_sk[i])), 6) for i in range(3)])

# ---- 验收 ----
assert abs(float(np.linalg.norm(M_sk[0])) - 1.0) < 1e-9, "每行应被 L2 归一化"

In [ ]:
sk = TfidfVectorizer(tokenizer=lambda t: t, lowercase=False, token_pattern=None)
M_ref = sk.fit_transform(docs).toarray()
# sklearn 内部词表顺序与我们的 vocab 不同，先对齐列
M_ref = M_ref[:, [sk.vocabulary_[w] for w in vocab]]

doc_freq_sy = sum(1 for d in docs if "渗油" in d)
diff_plain = float(np.abs(M_plain - M_ref).max())
diff_sk = float(np.abs(M_sk - M_ref).max())

print("教科书版  vs sklearn 最大绝对差: %.6f" % diff_plain)
print("sklearn口径 vs sklearn 最大绝对差: %.10f" % diff_sk)
print()
print("「渗油」的 df=%d，idf：教科书 %.6f | sklearn %.6f"
      % (doc_freq_sy, math.log(len(docs) / doc_freq_sy),
         math.log((1 + len(docs)) / (1 + doc_freq_sy)) + 1))

# ---- 验收 ----
assert abs(diff_plain - 0.322036) < 1e-5, "教科书公式与 sklearn 差得很远"
assert diff_sk < 1e-9, "对齐口径后必须几乎完全一致"

## 任务 6：n-gram 特征（sklearn 版）

`ngram_range` 会把特征数从 70 抬到 910。

In [ ]:
cv2 = CountVectorizer(tokenizer=lambda t: t, lowercase=False, token_pattern=None,
                      ngram_range=(1, 2))
M_bigram = cv2.fit_transform(docs)
feat2 = list(cv2.get_feature_names_out())

print("unigram+bigram 特征数:", M_bigram.shape[1], "（unigram 只有 %d）" % len(vocab))
print("非零元素:", M_bigram.nnz)
print("bigram 示例:", [f for f in feat2 if " " in f][:5])

# ---- 验收 ----
assert M_bigram.shape == (570, 910)
assert len([f for f in feat2 if " " in f]) == 840

## 任务 7：手写 n-gram 并逐格对账

两处要补：手写的 bigram「非零格子」集合，以及从 sklearn 特征名里筛出 bigram。

⚠️ **别以为 `feat[:70]` 就是那 70 个 unigram** —— sklearn 的输出是**按列名字典序**排的，
前 3 列直接就是台区编号那几项。筛 bigram 只能靠「名字里有空格」。

In [ ]:
uni_pairs = [(i, idx[w]) for i, d in enumerate(docs) for w in d]
bg_pairs = [(i, (a, b)) for i, d in enumerate(docs) for a, b in zip(d, d[1:])]
bg_keys = sorted({t for _, t in bg_pairs})
bg_pos = {t: j for j, t in enumerate(bg_keys)}

# TODO(6)：手写 bigram 的「非零格子」集合：去重后的 (文档号, 列号)
# 提示：集合推导式：{(i, bg_pos[t]) for i, t in bg_pairs}
# 提示：口径要和 unigram 的 set(uni_pairs) 对齐，两者相加才等于 sklearn 的 nnz
bg_cells = ____
uni_cells = set(uni_pairs)

print("手写 bigram 种类:", len(bg_keys), "| 出现次数（含重复）:", len(bg_pairs))
print("手写非零格子: unigram %d + bigram %d = %d"
      % (len(uni_cells), len(bg_cells), len(uni_cells) + len(bg_cells)))

M2_cv = CountVectorizer(tokenizer=lambda t: t, lowercase=False, token_pattern=None,
                        ngram_range=(1, 2))
M2 = M2_cv.fit_transform(docs)
feat2b = list(M2_cv.get_feature_names_out())

# TODO(7)：从 sklearn 的特征名里筛出 bigram（名字形如「词A 词B」，中间一个空格）
# 提示：列表推导式，条件是 " " in f
sk_bg = ____

print("sklearn 矩阵:", M2.shape, "| bigram 列:", len(sk_bg), "| nnz:", M2.nnz)
print("bigram 列集合与手写一致:", set(sk_bg) == {"%s %s" % (a, b) for a, b in bg_keys})
print("sklearn 前 3 列（注意这里就是它的输出顺序）:", feat2b[:3])

# ---- 验收 ----
assert len(bg_keys) == 840
assert len(bg_pairs) == 2861
assert len(uni_cells) == 3431
assert len(bg_cells) == 2861
assert len(uni_cells) + len(bg_cells) == 6292 == M2.nnz
assert set(sk_bg) == {"%s %s" % (a, b) for a, b in bg_keys}
assert feat2b[:3] == ["STATION_A_01", "STATION_A_01 一般", "STATION_A_01 严重"]

## 任务 8：字级 n-gram

两个坑：

1. 拼特征名时**两个汉字之间要加一个空格**（这是 sklearn 拼 n-gram 的约定）。
2. 筛 bigram 列用 `len(f) == 3`（2 个字 + 1 个空格），**不是 `len(f) == 2`** ——
   用错会得到 **0 个**，然后误以为 sklearn 不支持字级 n-gram。

In [ ]:
char_docs = [[c for c in texts[i] if HAN_RE.fullmatch(c)] for i in keep]
char_vocab = sorted({c for d in char_docs for c in d})

# TODO(8)：字级 bigram 的「非零格子」集合：相邻两字之间**加一个空格**再拼
# 提示：集合推导式：{(i, a + " " + b) for i, d in enumerate(char_docs) for a, b in zip(d, d[1:])}
# 提示：这个空格是 sklearn 拼 n-gram 特征名的约定，漏了就对不上账
c_bg = ____
c_uni = {(i, c) for i, d in enumerate(char_docs) for c in d}

print("字级：单字 %d 种 / 总字数 %d | bigram 出现 %d 次 / 去重格子 %d"
      % (len(char_vocab), sum(len(d) for d in char_docs),
         sum(len(d) - 1 for d in char_docs), len(c_bg)))

cvv = CountVectorizer(tokenizer=lambda t: t, lowercase=False, token_pattern=None,
                      ngram_range=(1, 2))
Mc = cvv.fit_transform(char_docs)
fc = list(cvv.get_feature_names_out())

# TODO(9)：按特征名长度筛出 unigram 列与 bigram 列
# 提示：单字列 len(f) == 1；bigram 名单字之间带空格，所以 len(f) == 3
fc_uni = ____
fc_bg = ____

print("sklearn 字级:", Mc.shape, "| unigram 列 %d | bigram 列 %d | nnz %d"
      % (len(fc_uni), len(fc_bg), Mc.nnz))
print("字级 bigram 示例（注意空格）:", fc_bg[:4])
print("手写 vs sklearn：unigram 列数一致 %s / bigram 列数一致 %s"
      % (len(c_uni) == len(fc_uni), len(c_bg) == len(fc_bg)))

cw = LogisticRegression(max_iter=1000)
cc = LogisticRegression(max_iter=1000)
s_word = cross_val_score(cw, M_bigram, labels, cv=5)
s_char = cross_val_score(cc, Mc, labels, cv=5)
print("分类 5 折：词级 %.6f | 字级 %.6f" % (s_word.mean(), s_char.mean()))

# ---- 验收 ----
assert len(char_vocab) == 125
assert sum(len(d) for d in char_docs) == 8578
assert len(c_uni) == 8440 and len(c_bg) == 7988
assert len(c_uni) + len(c_bg) == 16428 == Mc.nnz
assert Mc.shape == (570, 620)
assert len(fc_uni) == 125 and len(fc_bg) == 495
assert abs(s_word.mean() - 1.000000) < 1e-6
assert abs(s_char.mean() - 1.000000) < 1e-6

## 任务 9：n-gram 语言模型与困惑度

三处要补：① 训练集的 bigram 计数；② 测试集里「训练集没见过」的 bigram 条数；
③ MLE 条件概率与 Lidstone 平滑两个函数。

困惑度的公式已经写在 `perplexity()` 里，你只需要补概率。

⚠️ **δ 不是越小越好**：扫描结果是 **U 形**，最优值在 `0.15` 附近；
教科书上的「加一平滑」（δ=1.0）在这里比最优值差 **25%**。

In [ ]:
N_TR = 456
tr_docs, te_docs = docs[:N_TR], docs[N_TR:]
c1 = Counter(w for d in tr_docs for w in d)

# TODO(10)：训练集的 bigram 计数：把每篇的相邻对丢进 Counter
# 提示：Counter(t for d in tr_docs for t in zip(d, d[1:]))
# 提示：zip(d, d[1:]) 产出的就是 (w_i, w_{i+1}) 这些相邻对
c2 = ____
V = len(c1)

c2_te = Counter(t for d in te_docs for t in zip(d, d[1:]))
tot_te = sum(c2_te.values())

# TODO(11)：测试集里「训练集从未见过」的 bigram 出现次数
# 提示：sum(v for k, v in c2_te.items() if k not in c2)
miss = ____

print("训练集: unigram %d 种 / %d 个 token | bigram %d 种"
      % (len(c1), sum(c1.values()), len(c2)))
print("测试集: bigram 共 %d 个 | 其中训练集未见 %d (%.4f%%)"
      % (tot_te, miss, 100 * miss / tot_te))


def perplexity(test_docs, prob):
    """困惑度 PP = exp( -1/N * Σ ln P(w_i | w_{i-1}) )。任一项概率为 0 则整体 inf。"""
    log_sum, n, zeros = 0.0, 0, 0
    for d in test_docs:
        for a, b in zip(d, d[1:]):
            p = prob(a, b)
            n = n + 1
            log_sum = log_sum + (math.log(p) if p > 0 else 0.0)
            zeros = zeros + (1 if p <= 0 else 0)
    return (float("inf") if zeros else math.exp(-log_sum / n)), zeros


def mle(a, b):
    # TODO(12)：最大似然条件概率 P(b|a) = c(a,b) / c(a)
    # 提示：分子 c2.get((a, b), 0)；分母 c1[a]
    ____


def lidstone(a, b, delta):
    # TODO(13)：Lidstone 平滑：分子加 δ，分母加 δ·V（V 是训练集词表大小）
    # 提示：分母写成 c1[a] + delta * V，分子写成 c2.get((a, b), 0) + delta
    ____


print("")
print("均匀分布参照：PP = V =", V)
pp0, z0 = perplexity(te_docs, mle)
print("MLE 未平滑   测试 PP = %s（%d 个零概率）"
      % ("inf" if pp0 == float("inf") else "%.6f" % pp0, z0))

print("--- δ 扫描：困惑度先降后升（U 形）---")
scan = {}
for delta in (1.0, 0.5, 0.3, 0.2, 0.15, 0.1, 0.07, 0.05, 0.02, 0.01, 0.001):
    pp, _ = perplexity(te_docs, lambda a, b, delta=delta: lidstone(a, b, delta))
    scan[delta] = pp
    print("  δ=%-6s 测试 PP = %12.6f" % (delta, pp))
best_delta = min(scan, key=scan.get)
print("=> 最优 δ = %s，PP = %.6f" % (best_delta, scan[best_delta]))

pp_tr, _ = perplexity(tr_docs, mle)
print("过拟合的量化：MLE 在训练集 PP = %.6f，到测试集却是 inf" % pp_tr)

# ---- 验收 ----
assert len(c1) == 70 and len(c2) == 752
assert sum(c1.values()) == 2756
assert tot_te == 561 and miss == 102
assert abs(100 * miss / tot_te - 18.1818) < 1e-3
assert pp0 == float("inf") and z0 == 102
assert abs(scan[1.0] - 26.993018) < 1e-5
assert abs(scan[0.15] - 21.626352) < 1e-5
assert abs(scan[0.001] - 43.319158) < 1e-5
assert best_delta == 0.15
assert abs(pp_tr - 9.070455) < 1e-5

## 任务 10：n 越大越稀疏

统计测试集里「训练集没有」的 n-gram 出现次数，看未见率随 n 怎么变。

⚠️ 4-gram 的训练集种类会**比 3-gram 少** —— 不是统计错了，是稀疏让组合数本身收缩了。

In [ ]:
print("n 越大越稀疏（训练 %d 条 / 测试 %d 条）：" % (len(tr_docs), len(te_docs)))
print("%-8s %10s %10s %10s %10s" % ("n", "训练种类", "测试个数", "未见", "未见率"))
for n in (1, 2, 3, 4):
    ctr = Counter(t for d in tr_docs for t in zip(*(d[i:] for i in range(n))))
    cte = Counter(t for d in te_docs for t in zip(*(d[i:] for i in range(n))))
    # TODO(14)：统计测试集里「训练集没有」的 n-gram 出现次数
    # 提示：sum(v for k, v in cte.items() if k not in ctr)
    miss_n = ____
    total = sum(cte.values())
    print("%-8s %10d %10d %10d %9.4f%%"
          % ("%d-gram" % n, len(ctr), total, miss_n, 100 * miss_n / total))

# ---- 验收 ----
assert len(Counter(t for d in tr_docs for t in zip(*(d[i:] for i in range(1))))) == 70
assert len(Counter(t for d in tr_docs for t in zip(*(d[i:] for i in range(2))))) == 752
assert len(Counter(t for d in tr_docs for t in zip(*(d[i:] for i in range(3))))) == 1250
assert len(Counter(t for d in tr_docs for t in zip(*(d[i:] for i in range(4))))) == 1132

## 任务 11：共现矩阵

窗口内的词对累加；**别把自己和自己算进去**（`step` 已经帮你算好了）。

In [ ]:
WINDOW = 5
co = np.zeros((len(vocab), len(vocab)), dtype=np.float64)

for d in docs:
    ids = [idx[w] for w in d]
    for i, a in enumerate(ids):
        lo, hi = max(0, i - WINDOW), min(len(ids), i + WINDOW + 1)
        for j in range(lo, hi):
            step = 0 if j == i else 1
            # TODO(15)：把 step 累加进共现矩阵的 (a, ids[j]) 位置
            ____

print("共现矩阵形状:", co.shape, "| 非零:", int((co > 0).sum()), "| 总计数:", int(co.sum()))

# ---- 验收 ----
assert co.shape == (70, 70)
assert int(co.sum()) == 17348
assert int((co > 0).sum()) == 3256

## 任务 12：PPMI

PMI 后把负值截成 0。

In [ ]:
total = co.sum()
row_sum = co.sum(axis=1, keepdims=True)
col_sum = co.sum(axis=0, keepdims=True)

# TODO(16)：算 PMI，并把负值截断成 0（得到 PPMI）
# 提示：np.log((co * total) / (row_sum * col_sum)) 然后 np.maximum(pmi, 0)
pmi = ____
ppmi = ____

print("PPMI 非零:", int((ppmi > 0).sum()), "| 最大值 %.6f" % ppmi.max())

# ---- 验收 ----
assert int((ppmi > 0).sum()) == 2160
assert abs(float(ppmi.max()) - 1.931072) < 1e-5

## 任务 13：词向量与余弦最近邻

先降维，再补 `topn` 的相似度计算。

In [ ]:
svd_raw = TruncatedSVD(n_components=10, random_state=42)
emb_raw = svd_raw.fit_transform(co)
svd = TruncatedSVD(n_components=10, random_state=42)
emb = svd.fit_transform(ppmi)

print("词向量形状:", emb.shape)
print("原始计数版 解释方差比合计: %.6f" % svd_raw.explained_variance_ratio_.sum())
print("PPMI 版    解释方差比合计: %.6f" % svd.explained_variance_ratio_.sum())


def topn(word, matrix, k=3):
    v = matrix[idx[word]]
    # TODO(17)：算该词向量与所有词向量的余弦相似度
    # 提示：matrix @ v 除以两个范数之积（分母加 1e-12 防零）
    sims = ____
    order2 = np.argsort(-sims)
    return [(vocab[j], round(float(sims[j]), 6)) for j in order2 if j != idx[word]][:k]


for w in ["变压器", "渗油", "绝缘子", "发热"]:
    print(f"  「{w}」原始计数:", topn(w, emb_raw))
    print(f"  「{w}」PPMI   :", topn(w, emb))

# ---- 验收 ----
assert emb.shape == (70, 10)
assert abs(svd.explained_variance_ratio_.sum() - 0.488376) < 1e-5
assert topn("变压器", emb_raw, 1)[0][0] == "隔离开关"

## 任务 14：BPE 的两个基本操作

In [ ]:
BPE_CORPUS = ["变压器渗油", "变压器发热", "绝缘子破损", "绝缘子渗油", "断路器发热", "变压器"]
print("BPE 训练语料（6 条，故意很小，方便逐步观察）:", BPE_CORPUS)

In [ ]:
def count_pairs(words):
    """统计相邻符号对的加权频次。words: {符号元组: 频次}。"""
    pairs = Counter()
    for syms, freq in words.items():
        for i in range(len(syms) - 1):
            # TODO(18)：给这一对符号加上该词的频次
            ____
    return pairs


def merge_pair(words, pair):
    """把每个词里出现的 pair 合并成一个符号。"""
    merged = {}
    for syms, freq in words.items():
        out, i = [], 0
        while i < len(syms):
            take = 2 if tuple(syms[i:i + 2]) == pair else 1
            # TODO(19)：按 take 个符号拼成一个：2 个直接拼接，1 个原样
            ____
            ____
        merged[tuple(out)] = freq
    return merged

## 任务 15：BPE 训练（贪心合并）

In [ ]:
def bpe_train(corpus, n_merges):
    """从零训练 BPE：反复找出最高频的相邻符号对并合并。"""
    words = {}
    for w in corpus:
        words[tuple(w)] = words.get(tuple(w), 0) + 1

    merges = []
    for _ in range(n_merges):
        pairs = count_pairs(words)
        if not pairs:
            break
        # TODO(20)：挑出频次最高的符号对记为一次合并，并用它与语料做合并
        best = ____
        ____
        words = ____
    return merges, words


merges, final_words = bpe_train(BPE_CORPUS, 8)
print("合并序列（按发生顺序）:")
for step, pair in enumerate(merges, 1):
    print(f"  {step}. {pair[0]!r} + {pair[1]!r} -> {(pair[0] + pair[1])!r}")
print()
print("合并后每个词被切成:", {"/".join(k): v for k, v in final_words.items()})

# ---- 验收 ----
assert merges[0] == ("变", "压")
assert ("变压", "器") in merges, "「器」应跟着「变压」被合并出来"
assert len(merges) == 8

## 任务 16：BPE 编码

把每一轮的合并结果传给下一轮。

In [ ]:
def bpe_encode(word, merges):
    """按学到的合并规则，把一个新词切成子词。"""
    syms = tuple(word)
    for pair in merges:
        out, i = [], 0
        while i < len(syms):
            take = 2 if tuple(syms[i:i + 2]) == pair else 1
            out.append("".join(syms[i:i + take]))
            i += take
        # TODO(21)：把这一轮合并后的符号序列作为下一轮的输入
        syms = ____
    return list(syms)


for w in ["变压器渗油", "绝缘子发热", "母线生锈"]:
    print(f"  {w} -> {bpe_encode(w, merges)}")

print()
print("注意最后一个：语料里没出现过「母线」「生锈」，所以只能逐字切开。")
print("子词的价值就在这里 —— 没见过的词也能用已知符号拼出来，不会 OOV。")

# ---- 验收 ----
assert bpe_encode("变压器渗油", merges) == ["变压器渗油"]
assert bpe_encode("绝缘子发热", merges) == ["绝缘子", "发热"]
assert bpe_encode("母线生锈", merges) == ["母", "线", "生", "锈"]

## 任务 17：表示 × 分类器（含负对照）

三种表示都是满分；把标签随机打乱后掉到 0.19 —— 这才是关键结论。

In [ ]:
y = labels
sc_bow = cross_val_score(LogisticRegression(max_iter=1000), M_cv, y, cv=5)
sc_tfidf = cross_val_score(LogisticRegression(max_iter=1000), M_sk, y, cv=5)
sc_bigram = cross_val_score(LogisticRegression(max_iter=1000), M_bigram, y, cv=5)

print("预测「缺陷类型」（5 类，文本里确实有信号）:")
print("  词袋       mean %.6f  std %.6f" % (sc_bow.mean(), sc_bow.std()))
print("  TF-IDF     mean %.6f  std %.6f" % (sc_tfidf.mean(), sc_tfidf.std()))
print("  词袋 1-2gram mean %.6f  std %.6f" % (sc_bigram.mean(), sc_bigram.std()))

y_shuffled = np.random.default_rng(0).permutation(y)
sc_shuf = cross_val_score(LogisticRegression(max_iter=1000), M_cv, y_shuffled, cv=5)
print()
print("对照：把标签随机打乱后（文本里就没有这个信息了）:")
print("  词袋       mean %.6f   （≈ 1/5 = 0.2 的水平）" % sc_shuf.mean())

# ---- 验收 ----
assert sc_bow.mean() > 0.99, "信号词互不重叠，线性分类器应该满分"
assert sc_shuf.mean() < 0.35, "随机标签下准确率必须掉回瞎猜水平"

## 自查清单

- [ ] `CountVectorizer` 传了 `tokenizer` 就设了 `token_pattern=None`
- [ ] 词袋非零 3431、稀疏度 0.914010
- [ ] 教科书 TF-IDF 与 sklearn 差 **0.322036**
- [ ] 对齐口径后差 **0.0000000000**，且行范数为 1
- [ ] `ngram_range=(1, 2)` → 910 个特征
- [ ] 共现矩阵对角线上不计入自身
- [ ] PPMI 负值截成 0
- [ ] BPE 第一对合并是 `("变", "压")`
- [ ] 未登录词 `母线生锈` 也能被切成子词（不 OOV）
- [ ] 能解释"随机标签下准确率为什么掉到 0.19"